In [9]:
#%pip install requests pandas

In [45]:
import json
import time
import pandas as pd
import requests
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import urljoin

In [46]:
'''
https://map.geo.admin.ch/ gibt WGS 84 lat/long Links unten = south / west; rechts oben = north / east (in dieser reihenfolge)

Westlichster punkt in Luzern -> für Bounds west (7.838717)
47.227713, 7.838717

Östlichster Punkt Schwyz -> für BOUNDS east (9.005109)
47.170131, 9.005109

Südlichster Punkt im Kanton Uri -> für BOUNDS south (46.526306)
46.526306, 8.478879

Nördlichster Punkt Luzern -> für BOUNDS north (47.285069)
47.285069, 8.257735
'''

'\nhttps://map.geo.admin.ch/ gibt WGS 84 lat/long Links unten = south / west; rechts oben = north / east (in dieser reihenfolge)\n\nWestlichster punkt in Luzern -> für Bounds west (7.838717)\n47.227713, 7.838717\n\nÖstlichster Punkt Schwyz -> für BOUNDS east (9.005109)\n47.170131, 9.005109\n\nSüdlichster Punkt im Kanton Uri -> für BOUNDS south (46.526306)\n46.526306, 8.478879\n\nNördlichster Punkt Luzern -> für BOUNDS north (47.285069)\n47.285069, 8.257735\n'

In [47]:
BASE_URL = "https://flatfox.ch"

#Koordinaten eines grossen Rechtecks welches die gesamte Zentralschweiz abdecken soll
BOUNDS = {
    "west": 7.838717,
    "east": 9.005109,
    "south": 46.526306,
    "north": 47.285069,
}

MAX_COUNT = 400
BATCH_SIZE = 50

REQUEST_TIMEOUT = 30

# Beispielpause; keine bestätigte Flatfox-Abrufrate.
REQUEST_PAUSE = 2

# Grenzen der automatischen Gebietsaufteilung.
MAX_DEPTH = 8
MAX_SEARCH_REQUESTS = 1000

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "scripts" else Path.cwd()
OUTPUT_ROOT = PROJECT_DIR / "data" / "flatfox_data"

In [48]:
# HILFSFUNKTIONEN
def utc_now():
    return datetime.now(timezone.utc).isoformat()

def save_json(path, data):
    """Daten als UTF-8-JSON speichern."""
    with path.open("w", encoding="utf-8") as file:
        json.dump(
            data,
            file,
            ensure_ascii=False,
            indent=2,
        )

def get_list(session, path, params):
    """API-Anfrage ausführen und eine Liste von Inseraten erwarten."""
    time.sleep(REQUEST_PAUSE)

    response = session.get(
        urljoin(BASE_URL, path),
        params=params,
        timeout=REQUEST_TIMEOUT,
    )

    if not response.ok:
        raise requests.HTTPError(
            f"HTTP {response.status_code} bei {response.url}\n"
            f"Antwortauszug: {response.text[:1000]}",
            response=response,
        )

    data = response.json()

    if not isinstance(data, list):
        raise ValueError(
            f"{path}: Unerwartete Antwortstruktur. "
            "Erwartet wurde eine JSON-Liste."
        )

    if any(
        not isinstance(item, dict) or "pk" not in item
        for item in data
    ):
        raise ValueError(
            f"{path}: Mindestens ein Eintrag enthält keine Inserat-ID."
        )
    return data

def split_bounds(bounds):
    """Ein Rechteck in vier Teilrechtecke unterteilen."""
    middle_longitude = (bounds["west"] + bounds["east"]) / 2
    middle_latitude = (bounds["south"] + bounds["north"]) / 2

    return [
        {
            "west": west,
            "east": east,
            "south": south,
            "north": north,
        }
        for west, east in [
            (bounds["west"], middle_longitude),
            (middle_longitude, bounds["east"]),
        ]
        for south, north in [
            (bounds["south"], middle_latitude),
            (middle_latitude, bounds["north"]),
        ]
    ]

def attribute_names(record):
    """Gemeldete Ausstattungscodes eines Inserats auslesen."""
    attributes = record.get("attributes")

    if not isinstance(attributes, list):
        return None

    return {
        item["name"]
        for item in attributes
        if isinstance(item, dict)
        and isinstance(item.get("name"), str)
    }

def csv_value(value):
    """Listen und Dictionaries innerhalb von CSV-Zellen erhalten."""
    if isinstance(value, (list, dict)):
        return json.dumps(value, ensure_ascii=False)

    return value

In [49]:
# Main Hauptfunktion
def main():
    if not (-180 <= BOUNDS["west"] < BOUNDS["east"] <= 180 and
            -90 <= BOUNDS["south"] < BOUNDS["north"] <= 90):
        
        raise ValueError("Ungültige Grenzen in BOUNDS.")

    started_at = datetime.now(timezone.utc)

    # Eigener Ausgabeordner für jeden Durchlauf.
    run_name = started_at.strftime("%Y%m%dT%H%M%S_%fZ")
    output_dir = OUTPUT_ROOT / run_name
    output_dir.mkdir(parents=True, exist_ok=True)

    save_json(
        output_dir / "run_config.json",
        {
            "started_at": started_at.isoformat(),
            "bounds": BOUNDS,
            "max_count": MAX_COUNT,
            "batch_size": BATCH_SIZE,
            "max_depth": MAX_DEPTH,
            "max_search_requests": MAX_SEARCH_REQUESTS,
            "request_pause": REQUEST_PAUSE,
        },
    )

    with requests.Session() as session:

        # Pins suchen und volle Suchgebiete unterteilen
        print("Suche Inserate im definierten Kartenausschnitt ...")

        pending = [(BOUNDS.copy(), 0)]

        pins_by_id = {}
        search_log = []
        unresolved_areas = []

        request_number = 0

        while pending:
            if request_number >= MAX_SEARCH_REQUESTS:
                unresolved_areas.extend(
                    {
                        "bounds": bounds,
                        "depth": depth,
                        "reason": "request_budget_exhausted",
                    }
                    for bounds, depth in pending
                )
                break

            bounds, depth = pending.pop()

            tile_pins = get_list(
                session,
                "/api/v1/pin/",
                {
                    **bounds,
                    "max_count": MAX_COUNT,
                },
            )

            request_number += 1
            limit_reached = len(tile_pins) >= MAX_COUNT

            tile_metadata = {
                "fetched_at": utc_now(),
                "bounds": bounds,
                "depth": depth,
                "returned_count": len(tile_pins),
                "limit_reached": limit_reached,
            }

            # Jede Kartenantwort sofort vollständig speichern.
            save_json(
                output_dir / f"pins_tile_{request_number:04d}.json",
                {
                    **tile_metadata,
                    "pins": tile_pins,
                },
            )

            search_log.append(tile_metadata)

            # Auch Treffer aus übergeordneten Rechtecken behalten.
            # Wiederholte IDs werden hier zusammengeführt.
            for pin in tile_pins:
                pins_by_id[pin["pk"]] = pin

            print(
                f"Suchanfrage {request_number}: "
                f"Tiefe {depth}, "
                f"{len(tile_pins)} Pins, "
                f"{len(pins_by_id)} eindeutige IDs insgesamt"
            )

            if limit_reached:
                if depth >= MAX_DEPTH:
                    unresolved_areas.append({
                        "bounds": bounds,
                        "depth": depth,
                        "reason": "limit_reached_at_max_depth",
                    })
                else:
                    pending.extend(
                        (child, depth + 1)
                        for child in split_bounds(bounds)
                    )

        pins = list(pins_by_id.values())
        ids = sorted(pins_by_id)

        save_json(
            output_dir / "pins_raw.json",
            {
                "started_at": started_at.isoformat(),
                "search_finished_at": utc_now(),
                "search_bounds": BOUNDS,
                "max_count": MAX_COUNT,
                "request_count": request_number,
                "unresolved_areas": unresolved_areas,
                "search_log": search_log,
                "pins": pins,
            },
        )

        print(f"\nEindeutige Inserat-IDs: {len(ids)}")

        if unresolved_areas:
            print(
                f"ACHTUNG: {len(unresolved_areas)} Teilgebiete "
                "sind noch nicht vollständig abgearbeitet. "
                "Die Grenzen stehen in pins_raw.json. "
                "Gefundene Inserate werden trotzdem verarbeitet."
            )

        if not ids:
            print(f"Keine Inserate gefunden. Ausgabe: {output_dir.resolve()}")
            return pd.DataFrame()

        # Vollständige Inseratdetails abrufen
        listings = []
        batch_metadata = []

        total_batches = (
            len(ids) + BATCH_SIZE - 1
        ) // BATCH_SIZE

        for batch_number, start in enumerate(
            range(0, len(ids), BATCH_SIZE),
            start=1,
        ):
            batch_ids = ids[start:start + BATCH_SIZE]

            params = [
                ("limit", "0"),
                ("expand", "cover_image"),
            ] + [
                ("pk", str(listing_id))
                for listing_id in batch_ids
            ]

            print(
                f"Lade Detailgruppe {batch_number}/{total_batches} "
                f"mit {len(batch_ids)} IDs ..."
            )

            batch_data = get_list(
                session,
                "/api/v1/public-listing/",
                params,
            )

            fetched_at = utc_now()

            # Jede Detailantwort sofort speichern.
            save_json(
                output_dir / f"details_batch_{batch_number:04d}.json",
                {
                    "fetched_at": fetched_at,
                    "requested_ids": batch_ids,
                    "listings": batch_data,
                },
            )

            listings.extend(batch_data)

            batch_metadata.append({
                "batch": batch_number,
                "fetched_at": fetched_at,
                "requested_ids": batch_ids,
                "returned_count": len(batch_data),
            })

    # Vollständige Rohdaten speichern
    received_ids = {
        listing["pk"]
        for listing in listings
    }

    missing_ids = sorted(set(ids) - received_ids)
    unexpected_ids = sorted(received_ids - set(ids))

    save_json(
        output_dir / "listings_raw.json",
        {
            "started_at": started_at.isoformat(),
            "finished_at": utc_now(),
            "search_bounds": BOUNDS,
            "requested_ids": ids,
            "missing_ids": missing_ids,
            "unexpected_ids": unexpected_ids,
            "unresolved_areas": unresolved_areas,
            "batches": batch_metadata,
            "listings": listings,
        },
    )

    if not listings:
        print("Keine Inseratdetails erhalten.")
        print(f"Rohdaten gespeichert: {output_dir.resolve()}")
        return pd.DataFrame()

    # Alle Detailfelder in eine Tabelle übernehmen
    df = pd.json_normalize(listings, sep=".")

    df["collection_started_at"] = started_at.isoformat()

    df["source_url"] = [
        urljoin(BASE_URL, record["url"])
        if isinstance(record.get("url"), str)
        else None
        for record in listings
    ]

    # Ausstattung als zusätzliche Spalten ergänzen
    attribute_sets = [
        attribute_names(record)
        for record in listings
    ]

    all_attributes = sorted({
        name
        for names in attribute_sets
        if names is not None
        for name in names
    })

    for name in all_attributes:
        df[f"attribute_reported_{name}"] = pd.array(
            [
                name in names
                if names is not None
                else pd.NA
                for names in attribute_sets
            ],
            dtype="boolean",
        )

    # CSV mit allen Feldern speichern
    csv_df = df.copy()

    for column in csv_df.columns:
        csv_df[column] = csv_df[column].map(csv_value)

    csv_path = output_dir / "listings_all_fields.csv"
    
    csv_df.to_csv(
        csv_path,
        index=False,
        encoding="utf-8-sig",
    )

    print("\nDatenerhebung abgeschlossen.")
    print(f"Kartenanfragen: {request_number}")
    print(f"Gefundene eindeutige IDs: {len(ids)}")
    print(f"Detaildatensätze: {len(df)}")
    print(f"Eindeutige Inserate mit Details: {len(received_ids)}")
    print(f"Tabellenspalten: {len(df.columns)}")
    print(f"Nicht zurückgelieferte IDs: {missing_ids}")
    print(f"Unerwartet zurückgelieferte IDs: {unexpected_ids}")
    print(f"Unaufgelöste Teilgebiete: {len(unresolved_areas)}")
    print(f"Gefundene Ausstattungscodes: {all_attributes}")
    print(f"Ausgabeordner: {output_dir.resolve()}")

    return df

df = main()

Suche Inserate im definierten Kartenausschnitt ...
Suchanfrage 1: Tiefe 0, 400 Pins, 400 eindeutige IDs insgesamt
Suchanfrage 2: Tiefe 1, 400 Pins, 652 eindeutige IDs insgesamt
Suchanfrage 3: Tiefe 2, 400 Pins, 915 eindeutige IDs insgesamt
Suchanfrage 4: Tiefe 3, 103 Pins, 927 eindeutige IDs insgesamt
Suchanfrage 5: Tiefe 3, 42 Pins, 927 eindeutige IDs insgesamt
Suchanfrage 6: Tiefe 3, 317 Pins, 1004 eindeutige IDs insgesamt
Suchanfrage 7: Tiefe 3, 27 Pins, 1004 eindeutige IDs insgesamt
Suchanfrage 8: Tiefe 2, 14 Pins, 1004 eindeutige IDs insgesamt
Suchanfrage 9: Tiefe 2, 400 Pins, 1218 eindeutige IDs insgesamt
Suchanfrage 10: Tiefe 3, 338 Pins, 1405 eindeutige IDs insgesamt
Suchanfrage 11: Tiefe 3, 34 Pins, 1405 eindeutige IDs insgesamt
Suchanfrage 12: Tiefe 3, 194 Pins, 1483 eindeutige IDs insgesamt
Suchanfrage 13: Tiefe 3, 283 Pins, 1667 eindeutige IDs insgesamt
Suchanfrage 14: Tiefe 2, 92 Pins, 1696 eindeutige IDs insgesamt
Suchanfrage 15: Tiefe 1, 85 Pins, 1736 eindeutige IDs insg